# Phase 2: Data Preprocessing 

In [157]:
import os
import json
import numpy as np
from PIL import Image, ImageDraw
from shapely import wkt
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

os.makedirs("outputs", exist_ok=True)           

In [158]:
DATA_ROOT = "/kaggle/input/datasets/azra41221/train-ert/train/train/"
LABELS_DIR = os.path.join(DATA_ROOT, "labels")
TARGETS_DIR = os.path.join(DATA_ROOT, "targets")
IMAGES_DIR = os.path.join(DATA_ROOT, "images")

## Step 1: Mask Check (un-classified)
how 'un-classified' JSON polygons are represented in the ready PNG masks (0 or 1?)

In [159]:
post_label_files = [file for file in os.listdir(LABELS_DIR) if file.endswith("_post_disaster.json")]
print("Number of post label files:", len(post_label_files))

Number of post label files: 2799


### step 1.1: Find the first un-classified example

In [160]:
target_file = None
target_building = None

for file in post_label_files:
    with open(os.path.join(LABELS_DIR, file)) as f:
        data = json.load(f)
    for building in data["features"]["xy"]:
        if building["properties"].get("subtype") == "un-classified":
            target_file = file
            target_building = building
            break
    if target_file:
        break

print("File:", target_file)
print("Building uid:", target_building["properties"]["uid"])
print("Polygon (first 120 chars):", target_building["wkt"][:120])

File: hurricane-matthew_00000295_post_disaster.json
Building uid: 1dd51e6a-768a-49ef-8f2d-35bdf7297019
Polygon (first 120 chars): POLYGON ((742.505737444688 855.8178468614936, 750.6324867917134 867.8048021489221, 739.2550377033854 874.9157078291887, 


### step 1.2: Check the mask

In [161]:
mask_name = target_file.replace(".json", "_target.png")
mask = np.array(Image.open(os.path.join(TARGETS_DIR, mask_name)))
print("Mask shape:", mask.shape)

polygon = wkt.loads(target_building["wkt"])
canvas = Image.new("L", (1024, 1024), 0)
ImageDraw.Draw(canvas).polygon(list(polygon.exterior.coords), fill=1)
inside = np.array(canvas) == 1
print("Pixels inside polygon:", inside.sum())

values, counts = np.unique(mask[inside], return_counts=True)
print("Mask values inside polygon:", dict(zip(values.tolist(), counts.tolist())))

Mask shape: (1024, 1024)
Pixels inside polygon: 208
Mask values inside polygon: {0: 208}


Result: un-classified buildings are encoded as 0 (background) in the masks, so no special handling is needed. Mask values are 0-4 only.

## Step 2: Train/Test/Validation

### Step 2.1: Storing the pairs as a table

* one row per image pair: base name + disaster type

In [162]:
rows = []
for file in post_label_files:
    base_name = file.replace("_post_disaster.json","")
    disaster = base_name.split("_")[0]
    rows.append({"base_name": base_name,"disaster": disaster})

df = pd.DataFrame(rows)

print("Total pairs:", len(df))
print(df.head())
print()
print(df["disaster"].value_counts())

Total pairs: 2799
                    base_name           disaster
0  hurricane-matthew_00000295  hurricane-matthew
1         socal-fire_00000723         socal-fire
2   midwest-flooding_00000033   midwest-flooding
3         socal-fire_00000385         socal-fire
4         socal-fire_00000960         socal-fire

disaster
socal-fire             823
hurricane-michael      343
hurricane-harvey       319
hurricane-florence     319
midwest-flooding       279
hurricane-matthew      238
santa-rosa-wildfire    226
mexico-earthquake      121
palu-tsunami           113
guatemala-volcano       18
Name: count, dtype: int64


### Step 2.2: Splitting
Stratified by disaster name, so each split has the same disaster mix.

In [163]:
# 1. train_df
train_df, others_df = train_test_split(
    df,
    test_size = 0.2,
    stratify = df["disaster"], # for balanced distribution
    random_state = 42
)

# 2. val_df and test_df
val_df, test_df = train_test_split(
    others_df,
    test_size = 0.5,
    stratify = others_df["disaster"],
    random_state = 42
)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 2239
Validation: 280
Test: 280


In [164]:
# each disaster has a similar % in every split
summary = pd.DataFrame({
    "all": df["disaster"].value_counts(normalize=True) * 100,
    "train": train_df["disaster"].value_counts(normalize=True) * 100,
    "val": val_df["disaster"].value_counts(normalize=True) * 100,
    "test": test_df["disaster"].value_counts(normalize=True) * 100,
}).round(1)

print(summary)

                      all  train   val  test
disaster                                    
guatemala-volcano     0.6    0.7   0.4   0.7
hurricane-florence   11.4   11.4  11.4  11.4
hurricane-harvey     11.4   11.4  11.4  11.4
hurricane-matthew     8.5    8.5   8.6   8.6
hurricane-michael    12.3   12.2  12.5  12.1
mexico-earthquake     4.3    4.3   4.3   4.3
midwest-flooding     10.0   10.0  10.0  10.0
palu-tsunami          4.0    4.1   3.9   3.9
santa-rosa-wildfire   8.1    8.1   7.9   8.2
socal-fire           29.4   29.4  29.6  29.3


In [165]:
# Saving the splits
os.makedirs("outputs", exist_ok=True)

train_df.to_csv("outputs/train_split.csv", index=False)
val_df.to_csv("outputs/val_split.csv", index=False)
test_df.to_csv("outputs/test_split.csv", index=False)

print("Saved:", os.listdir("outputs"))

Saved: ['train_split.csv', 'test_split.csv', 'class_weights.json', 'val_split.csv']


## Step 3: Dataset and DataLoader

### Step 3.1: Dataset

1. basic version

In [166]:
class DamageDataset(Dataset):
    def __init__(self, df, images_dir, targets_dir):
        self.base_names = df["base_name"].tolist()
        self.images_dir = images_dir
        self.targets_dir = targets_dir
    
    def __len__(self):
        return len(self.base_names)
    
    def __getitem__(self, idx):
        base_name = self.base_names[idx]
        
        pre = np.array(Image.open(os.path.join(self.images_dir, f"{base_name}_pre_disaster.png")))
        post = np.array(Image.open(os.path.join(self.images_dir, f"{base_name}_post_disaster.png")))
        mask = np.array(Image.open(os.path.join(self.targets_dir, f"{base_name}_post_disaster_target.png")))
        
        return pre, post, mask

- test

In [167]:
dataset = DamageDataset(train_df, IMAGES_DIR, TARGETS_DIR)
print("Dataset size:", len(dataset))

pre, post, mask = dataset[0]
print("pre :", pre.shape)
print("post:", post.shape)
print("mask:", mask.shape)

Dataset size: 2239
pre : (1024, 1024, 3)
post: (1024, 1024, 3)
mask: (1024, 1024)


2. RGB | transpose

In [168]:
# ImageNet statistic - RGB
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class DamageDataset(Dataset):
    def __init__(self, df, images_dir, targets_dir):
        self.base_names = df["base_name"].tolist()
        self.images_dir = images_dir
        self.targets_dir = targets_dir

    def __len__(self):
        return len(self.base_names)

    def _load_image(self, path):
        image = np.array(Image.open(path))
        image = image.astype(np.float32) / 255.0        # 0-255 to 0-1
        image = (image - IMAGENET_MEAN) / IMAGENET_STD  
        image = np.transpose(image, (2, 0, 1))         
        return image

    def __getitem__(self, idx):
        base_name = self.base_names[idx]

        pre = self._load_image(os.path.join(self.images_dir, f"{base_name}_pre_disaster.png"))
        post = self._load_image(os.path.join(self.images_dir, f"{base_name}_post_disaster.png"))
        mask = np.array(Image.open(os.path.join(self.targets_dir, f"{base_name}_post_disaster_target.png")))

        return pre, post, mask

3. numpy to pytorch tensor

In [169]:
# ImageNet statistics - RGB
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class DamageDataset(Dataset):
    def __init__(self, df, images_dir, targets_dir):
        self.base_names = df["base_name"].tolist()
        self.images_dir = images_dir
        self.targets_dir = targets_dir

    def __len__(self):
        return len(self.base_names)

    def _load_image(self, path):
        image = np.array(Image.open(path))
        image = image.astype(np.float32) / 255.0     
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = np.transpose(image, (2, 0, 1))          
        return image

    def __getitem__(self, idx):
        base_name = self.base_names[idx]

        pre = self._load_image(os.path.join(self.images_dir, f"{base_name}_pre_disaster.png"))
        post = self._load_image(os.path.join(self.images_dir, f"{base_name}_post_disaster.png"))
        mask = np.array(Image.open(os.path.join(self.targets_dir, f"{base_name}_post_disaster_target.png")))

        pre_tensor = torch.from_numpy(pre)
        post_tensor = torch.from_numpy(post)
        mask_tensor = torch.from_numpy(mask).long()

        return pre_tensor, post_tensor, mask_tensor

In [170]:
dataset = DamageDataset(train_df, IMAGES_DIR, TARGETS_DIR)
print("Dataset size:", len(dataset))

pre, post, mask = dataset[0]
print("pre :", pre.shape, pre.dtype)
print("post:", post.shape, post.dtype)
print("mask:", mask.shape, mask.dtype)

Dataset size: 2239
pre : torch.Size([3, 1024, 1024]) torch.float32
post: torch.Size([3, 1024, 1024]) torch.float32
mask: torch.Size([1024, 1024]) torch.int64


### Step 3.2: DataLoader

In [171]:
# DataLoader for train
train_loader = DataLoader(
    dataset,
    batch_size = 4,
    shuffle = True,
    num_workers = 2
)

In [172]:
#DataLoader for validation
validation_dataset = DamageDataset(val_df, IMAGES_DIR, TARGETS_DIR)

val_loader = DataLoader(
    validation_dataset,
    batch_size = 4,
    shuffle = False,
    num_workers = 2,
)

## 4. Class weighting

In [173]:
class_counts = np.zeros(5, dtype = np.int64)

for base_name in tqdm(train_df["base_name"]):
    mask_path = os.path.join(TARGETS_DIR, f"{base_name}_post_disaster_target.png")
    mask = np.array(Image.open(mask_path))
    class_counts += np.bincount(mask.ravel(), minlength = 5)

class_names = ["background", "no-damage", "minor-damage", "major-damage", "destroyed"]
total = class_counts.sum()

for i, name in enumerate(class_names):
    print(f"{i} {name:13s} {class_counts[i]:>14,d}  ({class_counts[i] / total * 100:.2f}%)")

100%|██████████| 2239/2239 [00:27<00:00, 82.01it/s]

0 background     2,209,628,062  (94.12%)
1 no-damage        102,034,620  (4.35%)
2 minor-damage      12,878,592  (0.55%)
3 major-damage      15,828,441  (0.67%)
4 destroyed          7,391,949  (0.31%)


In [174]:
class_freq = class_counts / class_counts.sum()

class_weights = np.sqrt(1.0 / class_freq)

class_weights = class_weights / class_weights[0]

for i, name in enumerate(class_names):
    print(f"{i} {name:13s} weight: {class_weights[i]:.2f}")

0 background    weight: 1.00
1 no-damage     weight: 4.65
2 minor-damage  weight: 13.10
3 major-damage  weight: 11.82
4 destroyed     weight: 17.29


In [175]:
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(class_weights_tensor)

with open("outputs/class_weights.json", "w") as f:
    json.dump(
        {"class_names": class_names, "weights": class_weights.tolist()},  
        f,                                            
        indent=2,                     
    )
print("Saved: outputs/class_weights.json")

tensor([ 1.0000,  4.6536, 13.0986, 11.8152, 17.2894])
Saved: outputs/class_weights.json
